# BlueSky Data Report - Official Account History

## By William Adams, 10/20/2026

In this report, I would like to test a few hypotheses relating to BlueSky's official account and it's history. Bluesky is unique compared to other social medias for many reasons, one of the most interesting being it's very young age, especially considering the amount of people who use it. This means that posts from the entire history of the app are very easy to pull from, as even it's oldest posts are still comparatively young. 

With this in mind, the two hypotheses I want to test are:

1. I want to test if the newest posts from BlueSky have gotten more popular as more people have joined the app, as well as specifically how much more popular they are compared to their oldest posts.
   
2. I also want to compare the content of said posts with eachother in order to see what topics are common among both and which ones are exclusive. My hypothesis is that, given the history surrounding why BlueSky came to be, I will see more posts about politics and AI in the older posts compared to the newer posts.

In order to find the answers for these questions, I will use 2 API endpoints. The first is the "getProfile" endpoint, which I will use to access data about the BlueSky accounts, such as total number of posts and follower count. The second is the "getAuthorFeed" endpoint, which I will use to get data on the 30 most recent posts from BlueSky in one table, and the 30 oldest posts in another so that I may compare them.

There are some limits to the avaliable data to keep in-mind however. This method can only access posts that are currently still up. There may be posts made by BlueSky that were then deleted that the data will not show. The constantly-growing amount of views on every post is also difficult to display accurately, so I have elected to instead gage the popularity of each post using other, more stable metrics such are like and repost count. 

In [5]:

import requests
import pandas as pd
import time
from IPython.display import display


# Configuration


BASE_URL = "https://public.api.bsky.app/xrpc"
HANDLE = "bsky.app"

PROFILE_URL = f"{BASE_URL}/app.bsky.actor.getProfile"
FEED_URL = f"{BASE_URL}/app.bsky.feed.getAuthorFeed"

PAGE_SIZE = 100
POSTS_TO_COMPARE = 30

session = requests.Session()
session.headers.update({"Accept": "application/json"})


# Getting profile information


profile_response = session.get(
    PROFILE_URL,
    params={"actor": HANDLE},
    timeout=30
)
profile_response.raise_for_status()

profile = profile_response.json()
account_did = profile["did"]

# Data types that will be used for the BlueSky Account Profile section

profile_stats = {
    "Handle": profile.get("handle"),
    "Display Name": profile.get("displayName"),
    "Followers Count": profile.get("followersCount", 0),
    "Following Count": profile.get("followsCount", 0),
    "Total Posts Count": profile.get("postsCount", 0),
    "Profile URI": profile.get("did"),
}

print("Official BlueSky Account Profile")
print("-" * 40)

for metric, value in profile_stats.items():
    print(f"{metric}: {value}")


# Extracting post details


def get_post_details(feed_item):
    """Extract relevant fields from an original post."""

    post = feed_item.get("post", {})
    record = post.get("record", {})
    author = post.get("author", {})

    # Excluding reposts of other accounts' posts, only posts made by the official Bluesky Account will be displayed.
    if author.get("did") != account_did:
        return None

    # Using the post's original creation date.
    created_at = record.get("createdAt")
    indexed_at = post.get("indexedAt")
    timestamp = created_at or indexed_at

    return {
        "Post URI": post.get("uri"),
        "Post Date": timestamp,
        "Post Text": record.get("text", ""),
        "Likes": post.get("likeCount", 0),
        "Reposts": post.get("repostCount", 0),
        "Replies": post.get("replyCount", 0),
        "Quotes": post.get("quoteCount", 0),
    }


#  Retrieving 30 most recent posts


def get_recent_posts(number_of_posts=30):
    posts = []
    cursor = None
    seen_uris = set()

    while len(posts) < number_of_posts:
        params = {
            "actor": account_did,
            "filter": "posts_no_replies",
            "limit": PAGE_SIZE,
        }

        if cursor:
            params["cursor"] = cursor

        response = session.get(
            FEED_URL,
            params=params,
            timeout=30
        )
        response.raise_for_status()

        data = response.json()
        feed = data.get("feed", [])

        if not feed:
            break

        for item in feed:
            details = get_post_details(item)

            if details is None:
                continue

            uri = details["Post URI"]

            if uri not in seen_uris:
                posts.append(details)
                seen_uris.add(uri)

            if len(posts) >= number_of_posts:
                break

        cursor = data.get("cursor")

        if not cursor:
            break

    return posts


#  Retrieving 30 oldest posts


def get_oldest_posts(number_of_posts=30):
    """
    Retrieve the account's available feed history,
    then select its oldest original posts.
    """

    cursor = None
    oldest_posts = []
    seen_uris = set()

    while True:
        params = {
            "actor": account_did,
            "filter": "posts_no_replies",
            "limit": PAGE_SIZE,
        }

        if cursor:
            params["cursor"] = cursor

        response = session.get(
            FEED_URL,
            params=params,
            timeout=30
        )
        response.raise_for_status()

        data = response.json()
        feed = data.get("feed", [])

        if not feed:
            break

        for item in feed:
            details = get_post_details(item)

            if details is None:
                continue

            uri = details["Post URI"]

            if uri not in seen_uris:
                oldest_posts.append(details)
                seen_uris.add(uri)

        cursor = data.get("cursor")

        if not cursor:
            break

        # Brief pause between requests.
        time.sleep(0.1)

    # Sort from oldest to newest.
    oldest_posts.sort(
        key=lambda post: post["Post Date"] or ""
    )

    return oldest_posts[:number_of_posts]


#  Creating DataFrames


recent_data = get_recent_posts(POSTS_TO_COMPARE)
oldest_data = get_oldest_posts(POSTS_TO_COMPARE)

recent_posts_df = pd.DataFrame(recent_data)
oldest_posts_df = pd.DataFrame(oldest_data)

for dataframe in [recent_posts_df, oldest_posts_df]:
    if not dataframe.empty:
        dataframe["Post Date"] = (
            pd.to_datetime(
                dataframe["Post Date"],
                errors="coerce",
                utc=True
            )
            .dt.strftime("%Y-%m-%d")
        )

        # Display recent posts first in both DataFrames.
        dataframe.sort_values(
            "Post Date",
            ascending=False,
            inplace=True
        )

        dataframe.reset_index(drop=True, inplace=True)


#  Displaying reports


print("\n30 Most Recent Original Posts")
display(recent_posts_df)

print("\n30 Oldest Available Original Posts")
display(oldest_posts_df)

print("\nReport Summary")
print(f"Recent posts retrieved: {len(recent_posts_df)}")
print(f"Oldest posts retrieved: {len(oldest_posts_df)}")


#  Engagement comparison


comparison = pd.DataFrame({
    "Metric": [
        "Average Likes",
        "Average Reposts",
        "Average Replies",
        "Average Quotes",
    ],
    "30 Most Recent Posts": [
        recent_posts_df["Likes"].mean(),
        recent_posts_df["Reposts"].mean(),
        recent_posts_df["Replies"].mean(),
        recent_posts_df["Quotes"].mean(),
    ],
    "30 Oldest Posts": [
        oldest_posts_df["Likes"].mean(),
        oldest_posts_df["Reposts"].mean(),
        oldest_posts_df["Replies"].mean(),
        oldest_posts_df["Quotes"].mean(),
    ],
})

print("\nEngagement Comparison")
display(comparison.round(2))


Official BlueSky Account Profile
----------------------------------------
Handle: bsky.app
Display Name: Bluesky
Followers Count: 35332656
Following Count: 22
Total Posts Count: 886
Profile URI: did:plc:z72i7hdynmk6r22z27h6tvur

30 Most Recent Original Posts


,Post URI,Post Date,Post Text,Likes,Reposts,Replies,Quotes
0,at://did:plc:z72i7hdynmk6r22z27h6tvur/app.bsky...,2026-10-20,STREAMING LIVE NOW: stream.place/niedermeyer....,542,92,31,8
1,at://did:plc:z72i7hdynmk6r22z27h6tvur/app.bsky...,2026-10-19,"if you remember none pizza with left beef, it'...",1931,226,116,58
2,at://did:plc:z72i7hdynmk6r22z27h6tvur/app.bsky...,2026-10-17,no kings includes billionaires who think they’...,9135,1765,659,419
3,at://did:plc:z72i7hdynmk6r22z27h6tvur/app.bsky...,2026-10-16,"“Pro wrestling conversations, like sports on B...",1051,116,114,75
4,at://did:plc:z72i7hdynmk6r22z27h6tvur/app.bsky...,2026-10-16,v1.135 is live!\n\nYou can now browse quote po...,1440,205,179,80
5,at://did:plc:z72i7hdynmk6r22z27h6tvur/app.bsky...,2026-10-14,"Happy 100th birthday, Winnie-the-Pooh! \n\nDid...",2858,351,69,43
6,at://did:plc:z72i7hdynmk6r22z27h6tvur/app.bsky...,2026-10-14,saw some cool clouds today,7665,413,235,44
7,at://did:plc:z72i7hdynmk6r22z27h6tvur/app.bsky...,2026-10-12,This is NOT an ad for @alexgibneyfilm.bsky.soc...,3944,984,71,59
8,at://did:plc:z72i7hdynmk6r22z27h6tvur/app.bsky...,2026-10-09,a masterclass in alt text,3083,297,47,14
9,at://did:plc:z72i7hdynmk6r22z27h6tvur/app.bsky...,2026-10-05,Congratulations to Nobel Prize winner @deisser...,2380,200,42,8



30 Oldest Available Original Posts


,Post URI,Post Date,Post Text,Likes,Reposts,Replies,Quotes
0,at://did:plc:z72i7hdynmk6r22z27h6tvur/app.bsky...,2023-07-05,Bluesky is a public benefit corp with the miss...,1052,194,35,47
1,at://did:plc:z72i7hdynmk6r22z27h6tvur/app.bsky...,2023-07-04,📢 Version 1.38 is rolling out out now.\n\n• Mu...,1799,288,96,88
2,at://did:plc:z72i7hdynmk6r22z27h6tvur/app.bsky...,2023-07-03,"We just enabled account sign-ups again, so any...",4441,1121,241,164
3,at://did:plc:z72i7hdynmk6r22z27h6tvur/app.bsky...,2023-07-01,Our systems are currently experiencing some de...,5135,881,277,536
4,at://did:plc:z72i7hdynmk6r22z27h6tvur/app.bsky...,2023-07-01,📢 Version 1.36 is rolling out to the App & Pla...,1547,193,109,62
5,at://did:plc:z72i7hdynmk6r22z27h6tvur/app.bsky...,2023-07-01,We will temporarily be pausing Bluesky sign-up...,5389,1191,343,955
6,at://did:plc:z72i7hdynmk6r22z27h6tvur/app.bsky...,2023-07-01,The “Popular with Friends” feed is temporarily...,3225,250,359,269
7,at://did:plc:z72i7hdynmk6r22z27h6tvur/app.bsky...,2023-06-29,We’ve sent mixed messages over the past week a...,193,61,16,30
8,at://did:plc:z72i7hdynmk6r22z27h6tvur/app.bsky...,2023-06-29,We're publishing an update to our Terms of Ser...,273,78,10,22
9,at://did:plc:z72i7hdynmk6r22z27h6tvur/app.bsky...,2023-06-28,📢 Version 1.34 is now available!\n\n• Adding a...,605,130,33,39



Report Summary
Recent posts retrieved: 30
Oldest posts retrieved: 30

Engagement Comparison


,Metric,30 Most Recent Posts,30 Oldest Posts
0,Average Likes,4682.97,1109.43
1,Average Reposts,731.53,234.87
2,Average Replies,227.40,75.03
3,Average Quotes,260.97,107.57


This data report proves my first hypothesis correct, as we can see that the average ammount of likes, reposts, replies, and quotes in the 30 most recent posts are much higher than the same averages in the 30 oldest posts. This tracks with the original theory that the platform's increased userbase would case newer posts to be much more popular compared to older posts.

The second hypothesis, on the other hand, seems to have been proven false by the post text provided within both tables. Among the most recent posts, a fair few stick out as being overtly political, while the same cannot be said for the oldest posts. This is largly due to a factor that I had forgotten to consider relating to the older posts, that being the amount of them announcing new versions, bug fixes, and other software-related issues that come with the launching of a new, massive social media platform. As such, most posts about the political reasons involving the platform's creation are buried, meaning that there are actually far more of them in the 30 most recent posts instead.

This data report has been very illuminating on the history of BlueSky and just how much it has changed over time. However, as previously mentioned, there were some issues that I encountered with the data avaliable that could be improved in the future. For example, filters could be used in order to seperate the posts from those that are relevant and those that are not in a more refined manner. A fuller version of this Data Report in which every post that BlueSky has created could also lead to more accurate findings and a better visualization overall. I will make sure to explore alternate approaches in the future, in order to ensure that my data reports are the highest quality that they can be.